In [1]:
!pip install -q unsloth
!pip install -q langchain
!pip install -q chromadb
!pip install -q pypdf
!pip install -U langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.8/188.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 25.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.1/253.1 MB 6.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.1/107.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 MB 33.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 2.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 86.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6

In [2]:
import os
import torch
from torch.utils.data import Dataset
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from unsloth import FastLanguageModel
from transformers import AutoTokenizer, Trainer, TrainingArguments
import gc

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
gc.collect()
torch.cuda.empty_cache()

In [4]:
max_seq_length = 2048  
dtype = torch.float16  # T4 works best with float16
load_in_4bit = True  # 4-bit quantization for memory efficiency

In [5]:
pdf_path = "/kaggle/input/singapore-hist"  # Replace with your actual path
output_dir = "/kaggle/working/phi3_trained_model"

In [6]:
print(f"Loading model...")
# Model Selection - Using Phi-3 Mini for efficiency on T4 GPUs
model_name = "unsloth/Phi-3.5-mini-instruct"  # Efficiently runs on T4 GPUs

# Load the model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

Loading model...
==((====))==  Unsloth 2025.2.15: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    GPU: Tesla T4. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.37k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

In [7]:
print(f"Applying LoRA...")
# Apply LoRA - optimized for T4 GPUs
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,  
    bias = "none",     
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
)

Applying LoRA...


Unsloth 2025.2.15 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [8]:

def find_pdf_files(directory):
    pdf_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.lower().endswith('.pdf'):
                pdf_files.append(os.path.join(root, file))
    return pdf_files

# Data processing function for multiple PDFs
def process_multiple_pdfs(directory):
    print(f"Searching for PDFs in: {directory}")
    pdf_files = find_pdf_files(directory)
    
    if not pdf_files:
        print(f"No PDF files found in {directory}")
        return []
    
    print(f"Found {len(pdf_files)} PDF files.")
    all_text_chunks = []
    
    for pdf_file in pdf_files:
        try:
            print(f"Processing: {pdf_file}")
            pdf_loader = PyPDFLoader(pdf_file)
            documents = pdf_loader.load()
            
            if not documents:
                print(f"No content loaded from {pdf_file}")
                continue
            
            print(f"Loaded {len(documents)} pages from {pdf_file}")
            
            # Split into chunks - using RecursiveCharacterTextSplitter for better semantic splitting
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=512,
                chunk_overlap=50,
                separators=["\n\n", "\n", ".", " ", ""]
            )
            text_chunks = text_splitter.split_documents(documents)
            print(f"Split into {len(text_chunks)} text chunks")
            
            # Just use the text content
            text_content = [chunk.page_content for chunk in text_chunks]
            all_text_chunks.extend(text_content)
            
            # Free up memory after processing each PDF
            del documents
            del text_chunks
            gc.collect()
            
        except Exception as e:
            print(f"Error processing {pdf_file}: {str(e)}")
            import traceback
            traceback.print_exc()
    
    print(f"Total text chunks from all PDFs: {len(all_text_chunks)}")
    return all_text_chunks

# Create instruction format
def create_instruction_format(text):
    return f"<|system|>\nYou are an AI assistant that provides helpful, accurate information based on your training.\n<|user|>\nAnalyze and summarize this text passage: {text}\n<|assistant|>\nHere's my analysis and summary of the passage:"


In [9]:
class TextDataset(Dataset):
    def __init__(self, tokenized_texts):
        self.input_ids = tokenized_texts["input_ids"]
        self.attention_mask = tokenized_texts["attention_mask"]
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.input_ids[idx].clone()  # Use input as labels for instruction fine-tuning
        }

In [3]:

# Process multiple PDFs from the directory
pdf_directory = pdf_path
text_content = process_multiple_pdfs(pdf_directory)

if text_content:
    # Handle potentially large datasets by limiting size if needed
    if len(text_content) > 10000:  # Adjust this threshold based on your GPU memory
        print(f"Dataset is very large ({len(text_content)} chunks). Limiting to 10000 chunks to avoid OOM.")
        import random
        random.shuffle(text_content)  # Shuffle to get diverse content
        text_content = text_content[:10000]
    
    # Format each chunk with instruction format
    print("Formatting text chunks into instruction format...")
    formatted_texts = [create_instruction_format(text) for text in text_content]
    
    print(f"Tokenizing data...")
    # Tokenize the text for fine-tuning
    # Process in batches if the dataset is large
    batch_size = 1000  # Adjust based on your memory constraints
    all_input_ids = []
    all_attention_masks = []
    
    for i in range(0, len(formatted_texts), batch_size):
        print(f"Tokenizing batch {i//batch_size + 1}/{(len(formatted_texts)-1)//batch_size + 1}")
        batch = formatted_texts[i:i+batch_size]
        
        tokenized_batch = tokenizer(
            batch,
            truncation=True,
            padding="max_length",
            max_length=max_seq_length,
            return_tensors='pt'
        )
        
        all_input_ids.append(tokenized_batch["input_ids"])
        all_attention_masks.append(tokenized_batch["attention_mask"])
        
        # Free memory
        del batch
        del tokenized_batch
        gc.collect()
        torch.cuda.empty_cache()
    
    # Combine batches
    input_ids = torch.cat(all_input_ids, dim=0)
    attention_masks = torch.cat(all_attention_masks, dim=0)
    
    # Create a dictionary similar to tokenizer output
    tokenized_texts = {
        "input_ids": input_ids,
        "attention_mask": attention_masks
    }
    
    # Create the dataset
    dataset = TextDataset(tokenized_texts)
    print(f"Created dataset with {len(dataset)} examples")
    
    # Remove tokens to conserve GPU memory
    del tokenized_texts
    del all_input_ids
    del all_attention_masks
    del input_ids
    del attention_masks
    del formatted_texts
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("No data to process. Check your PDF directory and file content.")

NameError: name 'pdf_path' is not defined

In [2]:
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,  # Lower for T4 GPUs
    gradient_accumulation_steps=8,  # Increase this to compensate for smaller batch size
    save_steps=100,
    save_total_limit=2,
    learning_rate=1e-5,
    warmup_steps=100,
    logging_dir='./phi3_logs',
    logging_steps=10,
    fp16=True,  # Use mixed precision training
    dataloader_num_workers=2,  # Use multiple workers for data loading
    remove_unused_columns=False,  # Required for custom datasets
    ddp_find_unused_parameters=False,  # For distributed training
    report_to="none",  # Disable wandb reporting
    optim="adamw_torch",  # Use PyTorch's optimizer
)


NameError: name 'TrainingArguments' is not defined

In [ ]:

# Cell 10: Train Model
# Initialize trainer
if 'dataset' in locals():
    print(f"Starting training...")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
    )
    
    # Train the model
    trainer.train()
    
    print(f"Saving model to {output_dir}")
    # Save the fine-tuned model
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    print(f"Training complete! Model saved to {output_dir}")
else:
    print("Dataset not created. Cannot train model.")

In [ ]:
def test_model():
    print("\nTesting the trained model...")
    
    # Load the saved model
    test_model, test_tokenizer = FastLanguageModel.from_pretrained(
        output_dir,
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
    )
    
    # Sample query
    query = "What is the main topic discussed in the document?"
    
    # Generate a response
    prompt = f"<|system|>\nYou are an AI assistant that answers questions based on your training data.\n<|user|>\n{query}\n<|assistant|>\n"
    inputs = test_tokenizer(prompt, return_tensors="pt").to(test_model.device)
    
    with torch.no_grad():
        generated_ids = test_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    
    response = test_tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    
    # Extract just the assistant's response
    assistant_response = response.split('<|assistant|>\n')[-1]
    
    print(f"Query: {query}")
    print(f"Response: {assistant_response}")

# Test the model if it was trained successfully
if os.path.exists(output_dir):
    try:
        test_model()
    except Exception as e:
        print(f"Error testing model: {str(e)}")
else:
    print("Model not saved. Skipping testing.")


In [ ]:

print("\n" + "="*80)
print("NEXT STEPS")
print("="*80)
print("Your model is trained and saved to:", output_dir)
print("To use this model in another Kaggle notebook:")
print("""
1. Create a dataset from the model files:
   - Go to 'Data' tab in your profile
   - Click 'New Dataset'
   - Select all files from the output directory
   - Make it public or private as needed

2. In a new notebook, add the dataset and use this code to load the model:

```python
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    "/kaggle/input/your-model-dataset-name/phi3_trained_model",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True
)

def generate_response(query):
    prompt = f"<|system|>\\nYou are an AI assistant.\\n<|user|>\\n{query}\\n<|assistant|>\\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    
    response = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return response.split('<|assistant|>\\n')[-1]

# Test it
response = generate_response("What information can you provide from the document?")
print(response)
```
""")